# Gold v2 — 질문별 독립 검증 agent (최종 1회)

dev에서 고정해 GitHub에 게시한 `results/latest/gold_v2_verify_agent_dev/frozen_config.json`의 설정 **하나만** gold v2(246장 / 534문항: Q1·Q2 492 + Q3 42)에 돌립니다. 이 노트북에서는 도구·경로를 바꿀 수 없고, 같은 폴더를 다른 설정으로 다시 돌리면 실행이 거부됩니다.

- 독립 원칙은 dev와 같습니다: 문항마다 이미지·그 문항·자기 1단계 답만 사용, 모순은 최종 답 저장 후 지표로만 계산
- **1단계**: `lvlm_gold_v2_main_rerun_colab.ipynb`의 `normal_ref_retrieved_diffdate` 답 (같은 날 촬영 사진을 뺀 비슷한 정상 이미지)
- **원본만 답**(경로가 `disagreement`일 때): 재실행 폴더의 `image_only.jsonl`(MiniCPM) → 없으면 `results/latest/gold_v2_baseline/prediction_review.csv`
- **Q3**: dev에 위치 정답이 없어 dev에서 고르지 못했습니다. 같은 고정 설정을 적용하고(확대는 Q3에 적용 안 함) 보조 지표로 보고합니다. 정답 위치가 전부 가운데라 상수 기준선(항상 "결함 없음" 0.524 / 항상 "가운데" 0.476)과 함께 봅니다.

먼저 끝나 있어야 하는 것: gold 재실행 노트북(검색 참조 + `normal_ref_retrieved_diffdate`), dev agent 노트북의 `frozen_config.json` 게시.
GPU: **L4**. 호출 수 최대 534 × 모델 수(`all` 경로 기준, `disagreement`면 그보다 적음), 2모델이면 약 30~40분.
결과: `MyDrive/lvlm_insplad_results/gold_v2_verify_agent_gold/` → GitHub `results/latest/gold_v2_verify_agent_gold/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
RERUN_FOLDER = "lvlm_insplad_results/gold_v2_main_rerun"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_verify_agent_gold"
FROZEN_CONFIG = "results/latest/gold_v2_verify_agent_dev/frozen_config.json"   # GitHub에 게시된 dev 선택
MODELS = None   # None이면 dev에서 고른 모델 그대로. 4모델로 넓히려면 gold 실행 전에 정하세요
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RERUN_ROOT = DRIVE_ROOT / RERUN_FOLDER
STAGE1_DIR = RERUN_ROOT / "predictions"

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

frozen_path = REPO_DIR / FROZEN_CONFIG
if not frozen_path.is_file():
    raise FileNotFoundError("dev agent 노트북에서 frozen_config.json을 게시한 뒤 실행하세요: " + FROZEN_CONFIG)
frozen = json.loads(frozen_path.read_text(encoding="utf-8"))
print("고정 설정:", json.dumps(frozen, ensure_ascii=False, indent=1))
STAGE1_CONDITION, RULE = frozen["stage1_condition"], frozen["rule"]
RUN_MODELS = MODELS or frozen["dev_models"]
REFS_CSV = RERUN_ROOT / "refs_gold" / "retrieved_refs.csv"
IMAGE_ONLY_CSV = REPO_DIR / "results/latest/gold_v2_baseline/prediction_review.csv"
need = [REFS_CSV, IMAGE_ONLY_CSV] + [STAGE1_DIR / m / f"{STAGE1_CONDITION}.jsonl" for m in RUN_MODELS]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("먼저 gold 재실행 노트북(검색 조건)을 끝내세요. 없음: " + ", ".join(missing))
print("모델:", RUN_MODELS, "| 입력 확인 완료")

In [ ]:
# 실행 — 끊기면 이 셀만 다시 실행해도 완료된 호출은 재사용 (설정은 frozen_config에서만 옴)
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_verify_agent.py",
     "--qa-dir", QA_DIR, "--frozen-config", frozen_path, "--stage1-dir", STAGE1_DIR,
     "--stage1-condition", STAGE1_CONDITION, "--image-only-csv", IMAGE_ONLY_CSV,
     "--refs-csv", REFS_CSV, "--rule", RULE, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--models", *RUN_MODELS, "--checkpoint-every", str(CHECKPOINT_EVERY)])

In [ ]:
# 결과
pd.set_option("display.width", 240); pd.set_option("display.max_rows", 200)
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
print("[정확도]")
display(lb[["model_key","variant","q1_anomaly_f1","q2_defect_macro_f1","q1_accuracy","q2_accuracy","q3_accuracy",
            "rechecks","avg_lvlm_calls_per_q","recheck_latency_s"]].round(3))
print("\n[모순] 최종 답에서 따로 계산 (재확인·답 수정에는 쓰지 않음)")
display(lb[["model_key","variant","consistency_error","q1q2_contradictions_new","q1q2_contradictions_resolved",
            "q1q3_contradiction","q1q3_contradictions_new","q1q3_contradictions_resolved","joint_accuracy",
            "false_defect_rate"]].round(3))
display(pd.read_csv(RUN_ROOT / "per_question.csv").round(3))
sig = pd.read_csv(RUN_ROOT / "significance.csv")
sig["txt"] = sig.apply(lambda r: f"{r.delta:+.3f} [{r.ci_low:+.3f},{r.ci_high:+.3f}]" + ("*" if r.significant else ""), axis=1)
print("\n[1단계 대비] * = CI가 0을 포함하지 않음")
display(sig.pivot_table(index=["model_key","condition"], columns="metric", values="txt", aggfunc="first"))
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status), "(0이 아니면 실행 셀을 다시 실행)")

In [ ]:
if PUBLISH:
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", RUN_ROOT,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_verify_agent_gold",
         "--files", "leaderboard.csv", "per_question.csv", "metrics_by_asset.csv", "answers.csv",
         "significance.csv", "route_stats.json", "run_status.json", "run_metadata.json"], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_verify_agent_gold/")